In [14]:
import pandas as pd
import pickle
from itertools import product
from xgboost import XGBClassifier
from score_functions import get_probs, draw_precision_recall_curve, draw_confusion_matrix, print_scores
from model_pipeline import get_train_test_splits

In [ ]:
df = pd.read_csv("../data/processed/train_ohe.csv")
X_train, X_test, y_train, y_test = get_train_test_splits(df)
print(X_train.shape, y_train.shape)


(1037340, 83) (1037340,)


In [ ]:
def run_xgboost(model_df:pd.DataFrame, X_train, y_train, X_test, y_test):
    
    param_grid = {
    "eta": [0.05, 0.1],
    "n_estimators": [700, 1000],
    "max_depth": [7, 9],
    "gamma": [0.1, 0.2],
    "subsample": [0.9],
    "min_child_weight": [5],
    "colsample_bytree": [0.9],
    "reg_alpha": [0.2,0.3]
    }
    keys = list(param_grid.keys())
    model_num = 0
    for values in product(*(param_grid[k] for k in keys)):
        params = dict(zip(keys, values))
        model_name = f"xgboost_{model_num}"
        model_num += 1
        model = XGBClassifier(
            learning_rate=params["eta"],
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_child_weight=params["min_child_weight"],
            gamma=params["gamma"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_alpha=params["reg_alpha"],
            random_state=4,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        print(model_name)
        y_pred_prob = get_probs(model, X_test)
        roc_auc, pr_auc, best_recall = print_scores(y_test, y_pred_prob, target_precision=0.95)
        model_df.loc[len(model_df)] = [
            model_name,
            params["eta"],
            params["max_depth"],
            params["n_estimators"],
            params["gamma"],
            params["subsample"],
            params["min_child_weight"],
            params["colsample_bytree"],
            params["reg_alpha"],
            roc_auc,
            pr_auc,
            best_recall,
        ]
        # save
        pickle.dump(model, open("./models/vol_2/" + model_name+".pkl", "wb"))
        print("--------------------------------")
    print("training finished!")

In [ ]:
# model_df = pd.DataFrame(columns=[
#     "model_name",
#     "eta",
#     "max_depth",
#     "n_estimators",
#     "gamma",
#     "subsample",
#     "min_child_weight",
#     "colsample_bytree",
#     "reg_alpha",
#     "roc_auc",
#     "pr_auc",
#     "best_recall",
# ])


In [24]:
run_xgboost(model_df, X_train, y_train, X_test, y_test)

xgboost_0
roc_auc: 0.9991701713971728, precision recall score: 0.950050161572823, best recall at 0.95 precision: 0.8361092604930047
--------------------------------
xgboost_1
roc_auc: 0.999179398576726, precision recall score: 0.9502274391368661, best recall at 0.95 precision: 0.8347768154563624
--------------------------------
xgboost_2
roc_auc: 0.9992479835744524, precision recall score: 0.9524332410007978, best recall at 0.95 precision: 0.844103930712858
--------------------------------
xgboost_3
roc_auc: 0.999253771556026, precision recall score: 0.952776866453716, best recall at 0.95 precision: 0.8447701532311792
--------------------------------
xgboost_4
roc_auc: 0.9993015482432114, precision recall score: 0.9544323414515336, best recall at 0.95 precision: 0.8507661558960693
--------------------------------
xgboost_5
roc_auc: 0.9992974914882692, precision recall score: 0.9547580634238036, best recall at 0.95 precision: 0.8507661558960693
--------------------------------
xgboost_6

In [25]:
model_df.to_csv("./models/xgboost_model_df_4.csv", index=False)

In [28]:
model_df.sort_values(by="best_recall", ascending=False)[:10]

,model_name,eta,max_depth,n_estimators,gamma,subsample,min_child_weight,colsample_bytree,reg_alpha,roc_auc,pr_auc,best_recall
1262,xgboost_93,0.10,9,1000,0.1,0.9,5,0.9,0.3,0.999275,0.957329,0.852099
1254,xgboost_85,0.10,9,700,0.1,0.9,5,0.9,0.3,0.999299,0.957614,0.852099
1249,xgboost_80,0.10,7,700,0.1,0.9,5,0.9,0.1,0.999358,0.957908,0.850766
1270,xgboost_5,0.01,9,1200,0.2,0.9,5,0.9,0.2,0.999297,0.954758,0.850766
1269,xgboost_4,0.01,9,1200,0.1,0.9,5,0.9,0.2,0.999302,0.954432,0.850766
1256,xgboost_87,0.10,9,700,0.3,0.9,5,0.9,0.3,0.999262,0.958315,0.850766
1248,xgboost_79,0.10,9,500,0.3,0.9,5,0.9,0.3,0.999269,0.958290,0.850100
1275,xgboost_10,0.10,9,1200,0.1,0.9,5,0.9,0.2,0.999324,0.957920,0.850100
1257,xgboost_88,0.10,7,1000,0.1,0.9,5,0.9,0.1,0.999310,0.957820,0.849434
1271,xgboost_6,0.10,9,800,0.1,0.9,5,0.9,0.2,0.999317,0.957810,0.849434
